[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohsennasab/python-fundamentals-hh/blob/main/notebooks/10_snow-data/10_03_snodas_snow_analysis_mapping.ipynb)

# Module 10, Lesson 3: NOAA/NOHRSC SNODAS 1 km Analysis, Mapping, and UA Comparison
## Reconciling the 2013 spatial shift, analyzing snowmelt runoff (`1044`), and comparing SNODAS against Lesson 2's UA 800 m results

In this lesson, you analyze NOAA/NOHRSC **SNODAS (`G02158`)** snow water equivalent (`1034`) and 24-hour snowmelt runoff (`1044`) across the **Minnesota River Basin (HUC 4 `0702`, 44,050.3 km²)** and its **12 HUC-8 subbasins** during **WY2023 (`2022-11-01` to `2023-05-15`)**. You snap pre- and post-October 1, 2013 SNODAS grids onto one shared `EPSG:5070` (`1,000 m`) grid, mask snow-free `-9999` land in `1044` using `SWE`, map basin and subbasin snowmelt, and compare SNODAS against Lesson 2's UA 800 m exports (`SNODAS - UA`).

### Module 10 at a Glance

| Lesson | Notebook | Focus in Minnesota River HUC 4 (`0702`) |
|---|---|---|
| 1 | `10_01_downloading_snow_data.ipynb` | Live USGS WBD, UA 800 m NetCDF, and SNODAS raw binary `.tar` downloads, format inspection, GeoTIFF stack creation |
| 2 | `10_02_ua_snow_analysis_mapping.ipynb` | UA 800 m `EPSG:5070` reprojection, resampling tests, 196-day HUC-4 and 12-HUC-8 SWE, depth, and volume maps |
| 3 (this notebook) | `10_03_snodas_snow_analysis_mapping.ipynb` | SNODAS 1 km `2013-10-01` grid reconciliation, SWE and `1044` melt runoff analysis, maps, and `SNODAS - UA` comparison |

## Lesson Information

Lesson Level: Intermediate

Purpose: Equip H&H engineers to analyze, check, reproject, and map NOAA/NSIDC SNODAS daily SWE and snowmelt runoff, and compare SNODAS against UA 800 m outputs on a shared equal-area grid.

### Learning Objectives

By the end of this lesson, you will be able to:

1. Align pre- and post-October 1, 2013 SNODAS grids onto one snapped `EPSG:5070` (`1,000 m`) equal-area grid.
2. Apply product scale factors (`1034` SWE `* 1.0` mm, `1044` Melt `* 0.01` mm/day) and replace `-9999` snow-free land values inside `1044` with `0.0 mm/day` using valid `1034` land pixels.
3. Account for documented SNODAS historical anomalies (`2003 to 2004`, `G02158_missing_files.txt`, `2013-10-01` shift, `2014 to 2019` Zero Repair Mask, `2018` NWP switch, and the `32,767` alpine cap).
4. Compute 196-day SWE, 24-hour snowmelt runoff, and cumulative melt across HUC-4 and HUC-8 subbasins.
5. Quantify correlation, spatial bias, and subbasin differences (`SNODAS - UA`) against Lesson 2's UA 800 m results.

### Prerequisites

- Lesson 1 (`10_01_downloading_snow_data.ipynb`) and Lesson 2 (`10_02_ua_snow_analysis_mapping.ipynb`)
- Comfort with `rasterio`, `geopandas`, NumPy arrays, and `pandas`

### Inputs and Outputs

| Role | File | Notes |
|---|---|---|
| Input (from Lesson 1 / `data/`) | `minnesota_river_wbd_0702.gpkg`, `snodas_mn_river_wy2023_{swe,melt}.tif` | USGS WBD and 196-band native `EPSG:4326` SNODAS stacks |
| Input (2013 shift test) | `snodas_20130930_swe_subset.tif`, `snodas_20131001_swe_subset.tif`, `snodas_2013_shift_headers.csv` | Pre/post-shift subsets and parsed `.txt` header records |
| Input (from Lesson 2 / `data/`) | `ua_800m_mn_river_wy2023_swe.tif`, Lesson 2 UA CSVs (or `ua_800m_*_summary.csv` fallback) | Used for pixel and subbasin cross-comparison |
| Output | `module10_outputs/snodas_wy2023_huc04_daily_timeseries.csv` | Daily HUC-4 SWE, Snow Melt Runoff, volume, and snow cover |
| Output | `module10_outputs/snodas_wy2023_huc08_daily_timeseries.csv` | Daily SWE and Snow Melt for all 12 HUC-8 subbasins |
| Output | `module10_outputs/snodas_wy2023_vs_ua_comparison.csv` | 12-subbasin Peak SWE comparison table (`SNODAS - UA`) |
| Output | `module10_outputs/snodas_wy2023_peak_swe_5070_1000m.tif` | Clipped peak-date SNODAS SWE GeoTIFF in `EPSG:5070` (`1,000 m`) |
| Output (figures) | `module10_outputs/snodas_fig1_*.png` to `snodas_fig4_*.png` | Four 300 dpi figures formatted in Arial or Liberation Sans |

## Using AI in This Lesson

When asking an AI assistant to analyze SNODAS data, require it to check scale factors, land masks, and missing-day errata against printed `.txt` headers.

Keep three habits in Lesson 3:

1. Verify scale factors explicitly (`1034` SWE stores `m * 1,000` so `raw * 1.0 = mm`, whereas `1044` Melt stores `m * 100,000` so `raw * 0.01 = mm/day`).
2. Check how `-9999` snow-free land in `1044` is handled, because leaving `-9999` as `NaN` drops zero-melt land cells and inflates basin averages.
3. Keep missing dates from `G02158_missing_files.txt` explicit as `NaN` instead of filling outages with `0.0`.

Starter questions to try with your AI assistant:

- "Look at the parsed `2013-09-30` and `2013-10-01` header bounds. Compute the exact grid shift in meters at `44.8°N` and explain how snapping both dates to one `EPSG:5070` `1,000 m` grid aligns them pixel-for-pixel."
- "In SNODAS, `1034` (SWE) is a `06:00 UTC` snapshot while `1044` (Snow Melt) is a `06:00-06:00 UTC` 24-hour sum. Why can cumulative seasonal `1044` melt exceed single-day Peak SWE?" 

## Part 1: Overview of SNODAS Modeling Physics and Historical Archive Shifts

### How NOAA/NOHRSC Builds SNODAS

1. Hourly weather forcing downscales Numerical Weather Prediction (NWP) temperature, humidity, wind, precipitation, and TOPORAD solar radiation (`RUC2` through Sep 2018, `RAP` and `HRRR` from Oct/Nov 2018 onward).
2. A multi-layer energy and mass balance model simulates snowpack layering, cold content, surface/blowing sublimation (`1050`, `1039`), and basal snowmelt runoff (`1044`).
3. Once daily at `06:00 UTC` (`00:00 CST` / `01:00 CDT` in Minnesota), NOHRSC assimilates satellite snow-covered area, airborne gamma radiation flight lines (dense coverage across the Upper Midwest), and ground measurements.

### Timeline of Documented SNODAS Anomalies (`G02158`)

| Date / Period | Documented Issue (NSIDC User Guide rev. Apr 2025 and `G02158_missing_files.txt`) | Engineering Action |
|---|---|---|
| `2003-09-30` to `2004-09-30` | Pre-routine assimilation period (routine daily assimilation started Oct 2004, snowpack temp `1038` absent before `2004-02-21`) | Start multi-year climatologies at WY2005 (`2004-10-01`) |
| `2003 to 2012` (18 masked days) | 18 dates missing all masked files (including `2007-02-14`, `2007-03-26`, `2008-03-13`, `2012-12-20`), `2003-10-30` missing SWE, `2005-07-31` SWE erroneously all zeros | Keep missing days as `NaN` and flag any short linear interpolation |
| `2013-10-01` | Grid cell centers shifted `+0.00041667°` (`1/2400°`, ~33 m east, ~46 m north in MN), and `.txt` headers truncate bounds to 15 digits | Assign rational pre/post-2013 bounds by date and reproject onto one snapped `EPSG:5070` grid |
| `2014-10-09` to `2019-10-10` | Land/water mask bug pinned 126,950 coastal and lakeshore cells at `0.0` SWE and Depth until `2019-10-11` | Apply NSIDC's `SNODAS_Zero_Repair_Mask.tif` for lakeshore or coastal work in `2014 to 2019` |
| `Sep to Nov 2018` | Driving meteorology switched from `RUC2` to `RAP` and `HRRR` | Note forcing transition in pre/post-2018 trend checks |
| `July 2019` | Header file extension changed from `.hdr` to `.txt` so GDAL/ArcGIS stopped reading custom text as ENVI headers | Parse `.txt` headers directly in Python |
| Ongoing | Mountain cells where accumulation exceeds ablation grow until reaching the `int16` ceiling `32,767` (`32.767 m` SWE) | Screen alpine cells near `32,767` |

Primary sources:

- NSIDC `G02158` Landing Page and DOI: [nsidc.org/data/g02158/versions/1](https://nsidc.org/data/g02158/versions/1), [DOI: 10.7265/N5TB14TC](https://doi.org/10.7265/N5TB14TC)
- SNODAS User Guide (rev. April 2025): [g02158-v001-userguide_2_1.pdf](https://nsidc.org/sites/default/files/g02158-v001-userguide_2_1.pdf)
- Archive, Missing-Files Errata, and Zero Repair Mask: [G02158/masked/](https://noaadata.apps.nsidc.org/NOAA/G02158/masked/), [G02158_missing_files.txt](https://noaadata.apps.nsidc.org/NOAA/G02158/G02158_missing_files.txt), and [SNODAS_Zero_Repair_Mask.tif](https://noaadata.apps.nsidc.org/NOAA/G02158/ancillary/SNODAS_Zero_Repair_Mask.tif)

## Part 2: Workspace Setup, Colab File Locations, and Loading Data

### Where Files Are Located in Google Colab

When running in Google Colab, your active working directory is `/content/`. Click the **Files** folder icon on the left sidebar to browse folders visually:

1. Input files (`minnesota_river_wbd_0702.gpkg`, `snodas_mn_river_wy2023_swe.tif`, `snodas_mn_river_wy2023_melt.tif`, `snodas_2013*.tif/csv`, `ua_800m_mn_river_wy2023_swe.tif`) belong in `/content/data/` (or `/content/drive/MyDrive/module10_snow_data/` when `USE_GOOGLE_DRIVE = True`).
2. Outputs produced here (CSVs, Peak GeoTIFF, figures, and `lesson3_snodas_outputs.zip`) are saved in `/content/module10_outputs/` (or `/content/drive/MyDrive/module10_snow_data/outputs/` when `USE_GOOGLE_DRIVE = True`).
3. If you ran Lessons 1 and 2 in the same Colab runtime, or used `USE_GOOGLE_DRIVE = True`, Lesson 3 picks up Lesson 2's UA daily summary CSVs automatically. Otherwise it reads the reference UA summary CSVs from `data/`.

The first code cell checks and installs required Python libraries without relying on `osgeo.gdal`.

In [ ]:
import importlib.util  # Checks if a package can be imported without loading it yet
import subprocess      # Runs 'pip install' from Python when libraries are missing
import sys             # Provides the path to the active Python executable

# Map {import_name: pip_package_name}
REQUIRED_PACKAGES = {
    'rasterio': 'rasterio',
    'rioxarray': 'rioxarray',
    'geopandas': 'geopandas',
    'xarray': 'xarray',
}

# List comprehension keeps only libraries where find_spec(...) returns None (not installed)
missing = [
    pip_name for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing:
    print(f"Installing: {', '.join(missing)}")
    # '*missing' unpacks each library name as a CLI argument, and '-q' keeps pip output short
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
    print("Done.")
else:
    print("All required packages are already installed.")

### Importing Libraries and Configuring Arial Map Styling

This cell imports standard Python, numeric, raster, vector, and plotting packages, configures Arial (Liberation Sans fallback) at 300 dpi, and defines scale bar and north arrow map helpers.

In [ ]:
# Standard library
import datetime
import gzip
import importlib.util
import io
import math
import re
import tarfile
import time
import zipfile
from pathlib import Path

# Numerics and tables
import numpy as np
import pandas as pd
import requests

# Vector and geospatial
import geopandas as gpd
from pyproj import CRS, Geod

# Raster
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import Affine, from_bounds, from_origin
from rasterio.warp import Resampling, reproject, calculate_default_transform
from rasterio.windows import Window, from_bounds as window_from_bounds

# Plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle

# Configure Arial (Liberation Sans fallback on Linux/Colab) and high-resolution figure defaults
FONT_PREFERENCE = ['Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans']
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': FONT_PREFERENCE,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Scan installed fonts and pick the highest-priority font found
installed_fonts = {f.name for f in font_manager.fontManager.ttflist}
active_font = next((f for f in FONT_PREFERENCE if f in installed_fonts), 'DejaVu Sans')
print(f"Active figure font: {active_font}")

# White-to-blue-to-purple continuous color palette for SWE maps
SNOW_CMAP = LinearSegmentedColormap.from_list(
    'snodas_snow', ['#F4F8FB', '#D6EAF8', '#85C1E9', '#2E86C1', '#1B4F72', '#4A235A'], N=256
)
SNOW_CMAP.set_bad('white')  # Paint NaN cells white

# Warm amber-to-brown palette for snowmelt runoff rates and seasonal melt sums
MELT_CMAP = LinearSegmentedColormap.from_list(
    'snodas_melt', ['#FEF9E7', '#FAD7A0', '#EB984E', '#CA6F1E', '#873600'], N=256
)
MELT_CMAP.set_bad('white')

# Diverging red-white-blue palette centered at zero for difference maps
DIFF_CMAP = LinearSegmentedColormap.from_list(
    'snow_diff', ['#B03A2E', '#EC7063', '#F8F9F9', '#5DADE2', '#1B4F72'], N=256
)
DIFF_CMAP.set_bad('white')


def add_scalebar(ax, length_km: float = 50.0, location: tuple[float, float] = (0.04, 0.05), show_mid: bool = True):
    """Add an alternating black/white metric scale bar to an EPSG:5070 axis (in meters)."""
    # Read axis limits in meters and position scale bar by fractional offsets
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    x0 = xmin + location[0] * (xmax - xmin)
    y0 = ymin + location[1] * (ymax - ymin)
    bar_len = length_km * 1000.0
    bar_h = 0.015 * (ymax - ymin)
    half = bar_len / 2.0
    ax.add_patch(Rectangle((x0, y0), half, bar_h, facecolor='#1C2833', edgecolor='#1C2833', lw=0.8, zorder=10))
    ax.add_patch(Rectangle((x0 + half, y0), half, bar_h, facecolor='white', edgecolor='#1C2833', lw=0.8, zorder=10))
    ax.text(x0, y0 + bar_h * 1.4, '0', ha='center', va='bottom', fontsize=8, fontweight='bold', zorder=10)
    if show_mid:
        ax.text(x0 + half, y0 + bar_h * 1.4, f'{int(length_km / 2)}', ha='center', va='bottom', fontsize=8, zorder=10)
    ax.text(x0 + bar_len, y0 + bar_h * 1.4, f'{int(length_km)} km', ha='center', va='bottom', fontsize=8, fontweight='bold', zorder=10)


def add_north_arrow(ax, x_frac: float = 0.94, y_frac: float = 0.88):
    """Add a cartographic north arrow in axes fraction coordinates."""
    # Normalized axes-fraction coordinates keep the arrow fixed at the upper right
    ax.annotate(
        'N', xy=(x_frac, y_frac + 0.07), xytext=(x_frac, y_frac),
        xycoords='axes fraction', textcoords='axes fraction',
        ha='center', va='center', fontsize=10, fontweight='bold', color='#1C2833',
        arrowprops=dict(arrowstyle='-|>', lw=1.6, color='#1C2833', mutation_scale=13), zorder=10,
    )


def format_km_axes(ax):
    """Format EPSG:5070 meter axes as clean kilometers."""
    # Convert meter axis ticks to kilometers for readability
    fmt = mpl.ticker.FuncFormatter(lambda v, pos: f'{v / 1000:.0f}')
    ax.xaxis.set_major_formatter(fmt)
    ax.yaxis.set_major_formatter(fmt)
    ax.set_xlabel('Albers Easting (km, EPSG:5070)')
    ax.set_ylabel('Albers Northing (km, EPSG:5070)')


print(f"Versions: numpy {np.__version__}, pandas {pd.__version__}, "
      f"geopandas {gpd.__version__}, rasterio {rasterio.__version__}")

### Verifying or Uploading Lesson 3 Input Files

The cell below sets `DATA_DIR` and `OUTPUT_DIR`, prints their resolved disk paths, and checks that all required Lesson 3 inputs exist locally (or triggers one Colab upload dialog if missing).

In [ ]:
# Watershed, grid, and SNODAS server settings
TARGET_HUC4 = '0702'
TARGET_CRS = 'EPSG:5070'
TARGET_RES_M = 1000.0                 # Snapped 1 km equal-area grid (matches SHG1K)
BBOX_PAD_DEG = 0.15                   # 0.15-degree margin kept around HUC 4 from Lesson 1
SNODAS_NODATA = -9999
SNODAS_ROWS, SNODAS_COLS = 3351, 6935

SNODAS_BASE_URL = 'https://noaadata.apps.nsidc.org/NOAA/G02158'
SNODAS_MASKED_URL = f'{SNODAS_BASE_URL}/masked'
SNODAS_SHIFT_DATE = datetime.date(2013, 10, 1)
SNODAS_BOUNDS_PRE_2013 = (-124.73375000000000, 24.94958333333333, -66.94208333333333, 52.87458333333333)
SNODAS_BOUNDS_POST_2013 = (-124.73333333333333, 24.95000000000000, -66.94166666666667, 52.87500000000000)

# Key WY2023 SNODAS dates: Peak SWE, melt transition, and peak daily melt runoff
LIVE_SNODAS_DATES = ['2023-03-12', '2023-04-10', '2023-04-12']

# Match Lesson 1 and 2's Drive setting if running across separate Colab tabs
USE_GOOGLE_DRIVE = False
try:
    IN_COLAB = importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False

if USE_GOOGLE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/module10_snow_data')
    OUTPUT_DIR = DATA_DIR / 'outputs'
else:
    DATA_DIR = Path('data')
    OUTPUT_DIR = Path('module10_outputs')

# Create output and tar-cache directories if they do not exist yet
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LIVE_TAR_DIR = OUTPUT_DIR / 'snodas_live_tars'
LIVE_TAR_DIR.mkdir(parents=True, exist_ok=True)


def _normalize_upload_name(raw_name: str) -> str:
    # Strip browser duplicate suffixes like 'file (1).tif' -> 'file.tif'
    p = Path(raw_name)
    stem = re.sub(r'\s*\(\d+\)$', '', p.stem)
    return f"{stem}{p.suffix}"


def _locate_local_file(filename: str) -> Path | None:
    # Check DATA_DIR, ./data/, /content/, OUTPUT_DIR, and nearby subfolders
    candidates = [
        DATA_DIR / filename,
        Path('data') / filename,
        Path(filename),
        Path('/content/data') / filename,
        Path('/content') / filename,
        OUTPUT_DIR / filename,
        Path('module10_outputs') / filename,
        DATA_DIR / 'outputs' / filename,
    ]
    for path in candidates:
        if path.exists() and path.is_file() and path.stat().st_size > 0:
            return path
    if DATA_DIR.exists():
        for match in DATA_DIR.rglob(filename):
            if match.is_file() and match.stat().st_size > 0:
                return match
    for pat in (f'*/{filename}', f'*/*/{filename}'):
        for match in Path('.').glob(pat):
            if match.is_file() and match.stat().st_size > 0:
                return match
    return None


def _save_colab_uploads(uploaded: dict[str, bytes]) -> list[str]:
    # Save uploaded files or extract uploaded .zip archives into DATA_DIR
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    saved: list[str] = []
    for raw_name, content in uploaded.items():
        clean_name = _normalize_upload_name(raw_name)
        raw_cwd_path = Path(raw_name)
        if clean_name.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(content)) as zf:
                for info in zf.infolist():
                    if info.is_dir():
                        continue
                    member_name = Path(info.filename).name
                    if not member_name or member_name.startswith('.') or '__MACOSX' in info.filename:
                        continue
                    dest = DATA_DIR / _normalize_upload_name(member_name)
                    dest.write_bytes(zf.read(info.filename))
                    saved.append(dest.name)
        else:
            dest = DATA_DIR / clean_name
            dest.write_bytes(content)
            saved.append(dest.name)
        # Clean up Colab's duplicate root upload file outside DATA_DIR
        if raw_cwd_path.exists() and raw_cwd_path.is_file():
            dest_check = DATA_DIR / clean_name
            if raw_cwd_path.resolve() != dest_check.resolve():
                raw_cwd_path.unlink(missing_ok=True)
    return saved


def ensure_course_files(required_files: list[str]) -> dict[str, Path]:
    # Verify all required Lesson 3 inputs are available locally, or open a Colab upload dialog
    resolved: dict[str, Path] = {}
    missing: list[str] = []
    for fn in required_files:
        hit = _locate_local_file(fn)
        if hit is not None:
            resolved[fn] = hit
        else:
            missing.append(fn)
    if not missing:
        print(f"All {len(resolved)} Lesson 3 file(s) found locally in {next(iter(resolved.values())).parent}/.")
        return resolved
    if IN_COLAB:
        from google.colab import files as colab_files
        print("Missing Lesson 3 file(s):")
        for fn in missing:
            print(f"  - {fn}")
        print("\nSelect these file(s) from your data/ folder (or upload a .zip of data/):")
        uploaded = colab_files.upload()
        if uploaded:
            _save_colab_uploads(uploaded)
        for fn in list(missing):
            hit = _locate_local_file(fn)
            if hit is not None:
                resolved[fn] = hit
                missing.remove(fn)
    if missing:
        print("\nStill missing:")
        for fn in missing:
            print(f"  - {fn}")
        print(f"Run Lesson 1 (BUILD_FULL_SEASON=True) / Lesson 2 or place them in '{DATA_DIR}' and rerun.")
    else:
        print(f"All {len(resolved)} required Lesson 3 file(s) are ready.")
    return resolved


def find_course_file(filename: str, allow_upload: bool = True) -> Path:
    # Return the resolved local Path for a file, prompting in Colab if missing
    hit = _locate_local_file(filename)
    if hit is not None:
        return hit
    if IN_COLAB and allow_upload:
        from google.colab import files as colab_files
        print(f"File '{filename}' not found locally. Upload '{filename}' (or a .zip of data/):")
        uploaded = colab_files.upload()
        if uploaded:
            _save_colab_uploads(uploaded)
            hit = _locate_local_file(filename)
            if hit is not None:
                return hit
    raise FileNotFoundError(
        f"Could not find '{filename}' in '{DATA_DIR}', ./data/, or '{OUTPUT_DIR}'."
    )


def fetch_bytes(url: str, timeout: int = 90, retries: int = 3) -> bytes:
    # Download URL bytes with automatic exponential-backoff retries
    last_err = None
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=timeout)
            resp.raise_for_status()
            return resp.content
        except requests.RequestException as exc:
            last_err = exc
            if attempt < retries - 1:
                time.sleep(1.5 * (2 ** attempt))
    raise RuntimeError(f"Request failed for {url}: {last_err}")


LESSON3_DATA_FILES = [
    'minnesota_river_wbd_0702.gpkg',
    'snodas_mn_river_wy2023_swe.tif',
    'snodas_mn_river_wy2023_melt.tif',
    'snodas_20130930_swe_subset.tif',
    'snodas_20131001_swe_subset.tif',
    'snodas_2013_shift_headers.csv',
    'ua_800m_mn_river_wy2023_swe.tif',
]
# Fall back to reference UA CSV summaries in data/ if Lesson 2 was not run in this session
if _locate_local_file('ua_wy2023_huc04_daily_timeseries.csv') is None:
    LESSON3_DATA_FILES.append('ua_800m_mn_river_wy2023_basin_summary.csv')
if _locate_local_file('ua_wy2023_huc08_daily_timeseries.csv') is None:
    LESSON3_DATA_FILES.append('ua_800m_mn_river_wy2023_subbasin_summary.csv')

ensure_course_files(LESSON3_DATA_FILES)
print(f"Data directory   : {DATA_DIR.resolve()}")
print(f"Outputs directory: {OUTPUT_DIR.resolve()}")

## Part 3: Study Area and Reconciling the October 1, 2013 Spatial Grid Shift

On October 1, 2013, NOHRSC shifted SNODAS cell centers by `+0.00041667°` (`1/2400°`) so cell centers align with clean fractional latitude and longitude coordinates, and `.txt` headers also truncate coordinates to 15 digits (`-124.733333333328` compared to exact `-124.73333333333333`).

The cell below loads HUC 4 (`0702`) and its 12 HUC-8 subbasins, inspects `snodas_2013_shift_headers.csv`, measures the 2013 shift in meters at `44.8°N`, and snaps both `2013-09-30` and `2013-10-01` onto one shared `EPSG:5070` (`1,000 m`) grid.

In [ ]:
# Load USGS WBD HUC-4 basin and 12 HUC-8 subbasin boundaries
wbd_path = find_course_file('minnesota_river_wbd_0702.gpkg')
huc4_geo = gpd.read_file(wbd_path, layer='huc04')
huc8_geo = gpd.read_file(wbd_path, layer='huc08').sort_values('huc8').reset_index(drop=True)

# Reproject to equal-area EPSG:5070 (meters) to calculate true polygon areas
huc4_proj = huc4_geo.to_crs(TARGET_CRS)
huc8_proj = huc8_geo.to_crs(TARGET_CRS)
huc8_proj['short_id'] = huc8_proj['huc8'].str[-2:]
huc8_proj['area_km2'] = huc8_proj.geometry.area / 1e6
basin_area_km2 = float(huc4_proj.geometry.area.iloc[0] / 1e6)

# 0.15-degree padded WGS84 box matching Lesson 1's SNODAS slice window
w4, s4, e4, n4 = [float(v) for v in huc4_geo.to_crs('EPSG:4326').total_bounds]
PADDED_BBOX_4326 = (w4 - BBOX_PAD_DEG, s4 - BBOX_PAD_DEG, e4 + BBOX_PAD_DEG, n4 + BBOX_PAD_DEG)
print(f"Loaded HUC 4 {huc4_geo['huc4'].iloc[0]} ({huc4_geo['name'].iloc[0]}): {basin_area_km2:,.1f} km² across {len(huc8_proj)} HUC-8s")

# Inspect 2013-09-30 vs. 2013-10-01 bounds and header records
shift_csv = find_course_file('snodas_2013_shift_headers.csv')
shift_df = pd.read_csv(shift_csv)
print("\n2013-09-30 vs. 2013-10-01 SNODAS Header Comparison:")
print(shift_df[['date', 'bounds_regime', 'official_ulx', 'official_uly', 'header_x_offset', 'header_y_offset']].to_string(index=False))

# Measure the October 1, 2013 half-pixel grid shift in meters at Minnesota's midpoint latitude (44.8 N)
dx_deg = float(shift_df.loc[1, 'official_ulx'] - shift_df.loc[0, 'official_ulx'])
dy_deg = float(shift_df.loc[1, 'official_uly'] - shift_df.loc[0, 'official_uly'])
geod = Geod(ellps='WGS84')
_, _, shift_e_m = geod.inv(-95.0, 44.8, -95.0 + dx_deg, 44.8)
_, _, shift_n_m = geod.inv(-95.0, 44.8, -95.0, 44.8 + dy_deg)
print(f"\nShift magnitude: Δlon = {dx_deg:+.8f}° ({shift_e_m:.1f} m East at 44.8°N), "
      f"Δlat = {dy_deg:+.8f}° ({shift_n_m:.1f} m North) ~ 5% of a 1 km cell")


def build_snapped_grid(bounds, res_m: float):
    """Return (Affine, height, width) snapped cleanly to multiples of `res_m` in EPSG:5070."""
    minx, miny, maxx, maxy = bounds
    # Floor/ceil coordinates outward to whole 1,000 m multiples so all dates share one pixel origin
    x0 = math.floor(minx / res_m) * res_m
    y0 = math.floor(miny / res_m) * res_m
    x1 = math.ceil(maxx / res_m) * res_m
    y1 = math.ceil(maxy / res_m) * res_m
    return from_origin(x0, y1, res_m, res_m), int(round((y1 - y0) / res_m)), int(round((x1 - x0) / res_m))


# Create the common 1 km EPSG:5070 grid and HUC-4 interior pixel mask
dst_tr, dst_h, dst_w = build_snapped_grid(tuple(huc4_proj.total_bounds), TARGET_RES_M)
h4_mask = ~geometry_mask(huc4_proj.geometry, out_shape=(dst_h, dst_w), transform=dst_tr, invert=False)

# Reproject both 2013-09-30 (pre-shift) and 2013-10-01 (post-shift) onto dst_tr and check alignment
aligned_transforms = []
for fn in ('snodas_20130930_swe_subset.tif', 'snodas_20131001_swe_subset.tif'):
    p = find_course_file(fn)
    with rasterio.open(p) as src:
        raw = src.read(1)
        src_f = np.where(raw == SNODAS_NODATA, np.nan, raw.astype(np.float32))
        out = np.full((dst_h, dst_w), np.nan, dtype=np.float32)
        reproject(
            src_f, out, src_transform=src.transform, src_crs=src.crs,
            dst_transform=dst_tr, dst_crs=TARGET_CRS,
            src_nodata=np.nan, dst_nodata=np.nan, resampling=Resampling.bilinear,
        )
        aligned_transforms.append((fn, src.transform.c, src.transform.f, dst_tr, out.shape))

print(f"\nPre-2013 source UL : ({aligned_transforms[0][1]:.6f}, {aligned_transforms[0][2]:.6f})")
print(f"Post-2013 source UL: ({aligned_transforms[1][1]:.6f}, {aligned_transforms[1][2]:.6f})")
assert aligned_transforms[0][3] == aligned_transforms[1][3] and aligned_transforms[0][4] == aligned_transforms[1][4]
print(f"Aligned EPSG:5070 grid check: PASS (identical {dst_w}x{dst_h} grid @ {TARGET_RES_M:.0f} m)")

## Part 4: Equal-Area Reprojection (`EPSG:5070`, 1,000 m) and Resampling (State vs. Flux)

Across Minnesota (`43.2°N` to `46.4°N`), 30 arc-second `EPSG:4326` SNODAS cells vary from `~677 m` to `~641 m` east-west by `~926 m` north-south. Projecting onto snapped `EPSG:5070` (`1,000 m`, `1.00 km²` cells) standardizes area calculations and highlights two behaviors between `1034` SWE and `1044` Snow Melt Runoff:

- On steep melting snow edges, `cubic` spline interpolation undershoots below zero and creates negative SWE and negative melt cells.
- While `SWE (1034)` stores `0` on snow-free land, `Snow Melt Runoff (1044)` stores `-9999` on snow-free land. Passing `land_valid_mask = (swe_raw != -9999)` converts snow-free land melt to `0.0 mm/day` while keeping lakes and out-of-US water masked as `NaN`.

In [ ]:
# Read paths to the 196-band SNODAS SWE and Melt GeoTIFFs and extract the 2023-04-10 demo band
sn_swe_tif = find_course_file('snodas_mn_river_wy2023_swe.tif')
sn_melt_tif = find_course_file('snodas_mn_river_wy2023_melt.tif')

melt_demo_date = LIVE_SNODAS_DATES[1]  # 2023-04-10 active spring melt transition
with rasterio.open(sn_swe_tif) as s_src, rasterio.open(sn_melt_tif) as m_src:
    dates_str = list(s_src.descriptions)
    b_idx = dates_str.index(melt_demo_date) + 1  # Rasterio bands start at 1
    swe_demo_raw = s_src.read(b_idx)
    melt_demo_raw = m_src.read(b_idx)
    sn_crs, sn_tr = s_src.crs, s_src.transform


def reproject_snodas_slice(
    raw_arr: np.ndarray,
    src_crs,
    src_tr: Affine,
    scale_to_mm: float,
    resampling: Resampling,
    land_valid_mask: np.ndarray | None = None,
) -> np.ndarray:
    """Mask -9999, apply physical scale factor to mm, reproject buffered slice to EPSG:5070 1 km, and mask."""
    if land_valid_mask is None:
        # For SWE (1034): keep -9999 as NaN and multiply valid ints by scale_to_mm (1.0)
        src_f = np.where(raw_arr == SNODAS_NODATA, np.nan, raw_arr.astype(np.float32) * scale_to_mm)
    else:
        # For Melt (1044): snow-free land stores -9999, so convert valid land -9999 to 0.0 mm/day
        src_f = np.where(
            ~land_valid_mask,
            np.nan,
            np.where(raw_arr == SNODAS_NODATA, 0.0, raw_arr.astype(np.float32) * scale_to_mm),
        )
    dst = np.full((dst_h, dst_w), np.nan, dtype=np.float32)
    reproject(
        src_f, dst,
        src_transform=src_tr, src_crs=src_crs,
        dst_transform=dst_tr, dst_crs=TARGET_CRS,
        src_nodata=np.nan, dst_nodata=np.nan,
        resampling=resampling,
    )
    dst[~h4_mask] = np.nan  # Clip outside HUC-4 boundary after resampling
    return dst


METHODS = {
    'nearest': Resampling.nearest,
    'bilinear': Resampling.bilinear,
    'cubic': Resampling.cubic,
    'average': Resampling.average,
}

# Compare all 4 resampling options on both SWE (1034, * 1.0) and Melt Runoff (1044, * 0.01)
rows = []
demo_swe_grids = {}
demo_melt_grids = {}
swe_land_demo = (swe_demo_raw != SNODAS_NODATA)
for var_label, raw_arr, sc_mm, lmask, store in [
    ('SWE (1034, mm)', swe_demo_raw, 1.0, None, demo_swe_grids),
    ('Melt Runoff (1044, mm/d)', melt_demo_raw, 0.01, swe_land_demo, demo_melt_grids),
]:
    for m_name, alg in METHODS.items():
        g = reproject_snodas_slice(raw_arr, sn_crs, sn_tr, sc_mm, alg, land_valid_mask=lmask)
        store[m_name] = g
        v = g[~np.isnan(g)]
        # Total basin water volume: mean_m * valid_area_m^2 / 1e9 = km^3 (each 1 km cell = 1e6 m^2)
        vol_km3 = (float(v.mean()) / 1000.0) * (v.size * 1.0e6) / 1e9
        rows.append({
            'Product': var_label,
            'Method': m_name,
            'Valid cells': int(v.size),
            'Min (mm)': round(float(v.min()), 2),
            'Mean (mm)': round(float(v.mean()), 2),
            'Max (mm)': round(float(v.max()), 2),
            'Neg pixels (<0)': int((v < 0).sum()),
            'Basin vol (km³)': round(vol_km3, 4),
        })

sn_resamp_df = pd.DataFrame(rows)
print(f"Resampling Comparison on {melt_demo_date} (target {dst_w}x{dst_h} @ 1,000 m {TARGET_CRS}):\n")
print(sn_resamp_df.to_string(index=False))

# Figure 1: SNODAS Resampling Comparison + Cubic Undershoot Map (Arial)
fig, axes = plt.subplots(1, 5, figsize=(17.5, 4.3), sharey=True)
ext_5070 = [dst_tr.c, dst_tr.c + dst_w * dst_tr.a, dst_tr.f + dst_h * dst_tr.e, dst_tr.f]
vmax_demo = float(np.ceil(np.nanmax(demo_swe_grids['nearest']) / 10.0) * 10.0)

# Plot each of the 4 resampled SWE maps on a common color scale
for ax, m_name in zip(axes[:4], ['nearest', 'bilinear', 'cubic', 'average']):
    im = ax.imshow(demo_swe_grids[m_name], cmap=SNOW_CMAP, vmin=0, vmax=vmax_demo, extent=ext_5070)
    huc4_proj.boundary.plot(ax=ax, color='#1C2833', lw=1.0)
    v = demo_swe_grids[m_name][~np.isnan(demo_swe_grids[m_name])]
    ax.set_title(f"{m_name.title()}\nmean={v.mean():.1f}, min={v.min():.1f} mm", fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

fig.colorbar(im, ax=axes[:4], shrink=0.82, pad=0.02, label='SNODAS SWE (mm)')

# 5th panel shows Cubic minus Bilinear to reveal edge ringing
diff_cb = demo_swe_grids['cubic'] - demo_swe_grids['bilinear']
im_d = axes[4].imshow(diff_cb, cmap=DIFF_CMAP, vmin=-3.5, vmax=3.5, extent=ext_5070)
huc4_proj.boundary.plot(ax=axes[4], color='#1C2833', lw=1.0)
axes[4].set_title(f"Cubic - Bilinear\nmin={np.nanmin(diff_cb):.1f}, max={np.nanmax(diff_cb):+.1f} mm", fontsize=10)
axes[4].set_xticks([])
axes[4].set_yticks([])
fig.colorbar(im_d, ax=axes[4], shrink=0.82, pad=0.05, label='Difference (mm)')

fig.suptitle(f"SNODAS 1 km Resampling Behavior During Active Melt ({melt_demo_date})",
             fontsize=12.5, fontweight='bold', x=0.5, y=1.02)
fig.text(0.01, -0.05,
         f"Data: NOAA/NSIDC SNODAS G02158 | Target: EPSG:5070 (1,000 m) | Cubic created {int((demo_swe_grids['cubic'] < 0).sum()):,} negative-SWE pixels",
         fontsize=8.5, color='#566573')
fig1_path = OUTPUT_DIR / 'snodas_fig1_resampling_comparison.png'
fig.savefig(fig1_path, dpi=300)
plt.show()
print(f"Saved: {fig1_path}")

Ask your AI assistant: *"Look at the `Basin vol (km³)` column for Snow Melt Runoff (`1044`) across `nearest`, `bilinear`, `cubic`, and `average`. Why do all four methods stay nearly mass-conserving over a 44,050 km² basin even though `cubic` introduces negative pixel artifacts locally?"*

## Part 5: Full WY2023 SNODAS Season: SWE, Snowmelt Runoff, and 12 HUC-8 Subbasins

This section processes all 196 daily bands (`2022-11-01` to `2023-05-15`) of SNODAS `1034` SWE and `1044` Snow Melt Runoff, verifies `0 mm` agreement against the Peak SWE `.tar` when online, runs five automated QA/QC checks, summarizes all 12 HUC-8 subbasins, and creates Figure 2 (Peak SWE and Cumulative Melt maps) and Figure 3 (SWE/Melt hydrograph).

In [ ]:
def read_snodas_tar_swe(tar_path: Path, d: datetime.date, bbox: tuple):
    # Pick pre- or post-Oct 2013 bounds by date and crop product 1034 (SWE) directly from a .tar archive
    bounds = SNODAS_BOUNDS_PRE_2013 if d < SNODAS_SHIFT_DATE else SNODAS_BOUNDS_POST_2013
    full_tr = from_bounds(*bounds, SNODAS_COLS, SNODAS_ROWS)
    rw = window_from_bounds(*bbox, transform=full_tr)
    c0, r0 = max(0, math.floor(rw.col_off)), max(0, math.floor(rw.row_off))
    c1, r1 = min(SNODAS_COLS, math.ceil(rw.col_off + rw.width)), min(SNODAS_ROWS, math.ceil(rw.row_off + rw.height))
    tr_sub = rasterio.windows.transform(Window(c0, r0, c1 - c0, r1 - r0), full_tr)
    with tarfile.open(tar_path, 'r:*') as tf:
        for member in tf.getmembers():
            mname = Path(member.name).name
            if 'ssmv11034' in mname and mname.endswith('.dat.gz'):
                # Decompress gzip bytes and decode big-endian int16 ('>i2') across the CONUS grid
                raw = gzip.decompress(tf.extractfile(member).read())
                full_arr = np.frombuffer(raw, dtype='>i2').reshape(SNODAS_ROWS, SNODAS_COLS)
                return full_arr[r0:r1, c0:c1].astype(np.int16, copy=True), tr_sub
    raise RuntimeError(f"1034 SWE grid not found in {tar_path.name}")


# Read all 196 daily bands of SNODAS SWE and Melt into 3D NumPy arrays
with rasterio.open(sn_swe_tif) as s_src, rasterio.open(sn_melt_tif) as m_src:
    dates_str = list(s_src.descriptions)
    assert dates_str == list(m_src.descriptions), "SWE and Melt dates differ!"
    swe_raw_stack = s_src.read()
    melt_raw_stack = m_src.read()
    sn_crs, sn_tr = s_src.crs, s_src.transform

# Spot-check the 196-day GeoTIFF against a live (or cached) Peak SWE tarball when online
pk_d_live = LIVE_SNODAS_DATES[0]
pk_band_idx = dates_str.index(pk_d_live)
pk_tar_name = f"SNODAS_{pk_d_live.replace('-', '')}.tar"
pk_tar_path = _locate_local_file(pk_tar_name) or (LIVE_TAR_DIR / pk_tar_name)
try:
    if not pk_tar_path.exists() or pk_tar_path.stat().st_size == 0:
        d_obj = datetime.date.fromisoformat(pk_d_live)
        pk_tar_path.write_bytes(fetch_bytes(f"{SNODAS_MASKED_URL}/{d_obj:%Y}/{d_obj:%m_%b}/{pk_tar_name}", timeout=60, retries=2))
    live_swe, live_tr = read_snodas_tar_swe(pk_tar_path, datetime.date.fromisoformat(pk_d_live), PADDED_BBOX_4326)
    assert np.allclose(tuple(live_tr), tuple(sn_tr), atol=1e-7), "SNODAS transform mismatch!"
    max_diff = int(np.max(np.abs(live_swe.astype(np.int32) - swe_raw_stack[pk_band_idx].astype(np.int32))))
    assert max_diff == 0, f"Course GeoTIFF differs from live tar by {max_diff} mm!"
    print(f"Live tar vs. stack check on {pk_d_live}: PASS (max diff = {max_diff} mm)")
except Exception as err:
    print(f"Live SNODAS tar spot-check skipped offline ({err}); continuing with 196-day stack.")

# Precompute boolean masks for all 12 HUC-8 subbasins on the 1 km EPSG:5070 grid
h8_masks = [
    ~geometry_mask([geom], out_shape=(dst_h, dst_w), transform=dst_tr, invert=False)
    for geom in huc8_proj.geometry
]
cell_area_km2 = (TARGET_RES_M ** 2) / 1e6                # 1,000 m x 1,000 m = 1.00 km^2 per pixel
basin_area_m2 = float(huc4_proj.geometry.area.iloc[0])

# Preallocate 3D arrays (196 days, rows, cols) filled with NaN
n_days = len(dates_str)
swe_5070_stack = np.full((n_days, dst_h, dst_w), np.nan, dtype=np.float32)
melt_5070_stack = np.full((n_days, dst_h, dst_w), np.nan, dtype=np.float32)

sn_basin_rows = []
sn_sub_rows = []

# Loop through all 196 days: reproject SWE and Melt, then aggregate over HUC-4 and each HUC-8
for i, d_str in enumerate(dates_str):
    # Valid SWE pixels identify land cells where Melt -9999 represents 0.0 mm/day
    land_mask_i = (swe_raw_stack[i] != SNODAS_NODATA)
    sg = reproject_snodas_slice(swe_raw_stack[i], sn_crs, sn_tr, 1.0, Resampling.bilinear)
    mg = reproject_snodas_slice(melt_raw_stack[i], sn_crs, sn_tr, 0.01, Resampling.bilinear, land_valid_mask=land_mask_i)
    swe_5070_stack[i] = sg
    melt_5070_stack[i] = mg

    sv = sg[h4_mask]
    mv = mg[h4_mask]
    sv = sv[~np.isnan(sv)]
    mv = mv[~np.isnan(mv)]

    mean_swe = float(sv.mean()) if sv.size else 0.0
    mean_melt = float(mv.mean()) if mv.size else 0.0
    # Convert mean SWE (mm -> m) * watershed area (m^2) into km^3 and kAF
    vol_km3 = (mean_swe / 1000.0) * basin_area_m2 / 1e9
    vol_kaf = vol_km3 * 810.7132

    sn_basin_rows.append({
        'date': pd.to_datetime(d_str),
        'mean_swe_mm': round(mean_swe, 3),
        'mean_swe_in': round(mean_swe / 25.4, 4),
        'max_swe_mm': round(float(sv.max()), 2),
        'mean_melt_mm_day': round(mean_melt, 3),
        'snow_cover_pct': round(float((sv > 1.0).mean() * 100.0), 2),
        'swe_volume_km3': round(vol_km3, 4),
        'swe_volume_kaf': round(vol_kaf, 2),
        'valid_cells': int(sv.size),
        'valid_area_km2': round(sv.size * cell_area_km2, 2),
    })

    # Compute daily SWE and melt averages for each of the 12 HUC-8 subbasins
    for j, m_sub in enumerate(h8_masks):
        sv_s = sg[m_sub]
        mv_s = mg[m_sub]
        sv_s = sv_s[~np.isnan(sv_s)]
        mv_s = mv_s[~np.isnan(mv_s)]
        sub_swe = float(sv_s.mean()) if sv_s.size else 0.0
        sub_melt = float(mv_s.mean()) if mv_s.size else 0.0
        sub_area_m2 = float(huc8_proj.iloc[j].geometry.area)
        sub_kaf = ((sub_swe / 1000.0) * sub_area_m2 / 1e9) * 810.7132
        sn_sub_rows.append({
            'date': pd.to_datetime(d_str),
            'short_id': huc8_proj.iloc[j]['short_id'],
            'huc8': huc8_proj.iloc[j]['huc8'],
            'name': huc8_proj.iloc[j]['name'],
            'mean_swe_mm': round(sub_swe, 3),
            'mean_swe_in': round(sub_swe / 25.4, 4),
            'mean_melt_mm_day': round(sub_melt, 3),
            'swe_volume_kaf': round(sub_kaf, 2),
        })

sn_basin_df = pd.DataFrame(sn_basin_rows)
# Running seasonal sum of daily basin-mean melt runoff (mm)
sn_basin_df['cum_melt_mm'] = sn_basin_df['mean_melt_mm_day'].cumsum().round(2)
sn_sub_df = pd.DataFrame(sn_sub_rows)

# Run automated PASS/FAIL QA/QC checks before plotting
valid_area_pct = float(sn_basin_df['valid_area_km2'].iloc[0] / basin_area_km2 * 100.0)
qa_checks = [
    ('Target CRS is EPSG:5070', CRS.from_user_input(TARGET_CRS).to_epsg() == 5070, TARGET_CRS),
    ('Day count complete (196 days)', len(sn_basin_df) == 196, f"{len(sn_basin_df)} days"),
    ('Zero negative SWE pixels', bool(np.nanmin(swe_5070_stack) >= 0.0), f"min={np.nanmin(swe_5070_stack):.3f}"),
    ('Zero negative Melt pixels', bool(np.nanmin(melt_5070_stack) >= 0.0), f"min={np.nanmin(melt_5070_stack):.3f}"),
    ('Valid area >= 99%', valid_area_pct >= 99.0, f"{valid_area_pct:.2f}%"),
]
print("\nQA/QC Verification:")
for label, ok, detail in qa_checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label:<34s} -> {detail}")
    assert ok, f"QA check failed: {label}"

# Identify peak SWE day and peak daily melt day across the 196-day series
sn_pk_row = sn_basin_df.loc[sn_basin_df['mean_swe_mm'].idxmax()]
sn_melt_pk_row = sn_basin_df.loc[sn_basin_df['mean_melt_mm_day'].idxmax()]
print(f"\nSNODAS WY2023 Peak SWE  : {sn_pk_row['mean_swe_mm']:.1f} mm ({sn_pk_row['mean_swe_in']:.2f} in) on {sn_pk_row['date'].strftime('%Y-%m-%d')} | {sn_pk_row['swe_volume_kaf']:,.0f} kAF")
print(f"SNODAS WY2023 Peak Melt : {sn_melt_pk_row['mean_melt_mm_day']:.1f} mm/day on {sn_melt_pk_row['date'].strftime('%Y-%m-%d')} | Cum melt = {sn_basin_df['cum_melt_mm'].iloc[-1]:.1f} mm")

# Build summary table of peak SWE, peak melt rate, and total cumulative melt for each HUC-8
sub_rows = []
for h8, grp in sn_sub_df.groupby('huc8', sort=True):
    grp = grp.sort_values('date').reset_index(drop=True)
    pk = grp.loc[grp['mean_swe_mm'].idxmax()]
    mlt_pk = grp.loc[grp['mean_melt_mm_day'].idxmax()]
    sub_rows.append({
        'ID': pk['short_id'],
        'HUC-8': h8,
        'Subbasin': pk['name'],
        'Peak SWE (mm)': round(pk['mean_swe_mm'], 1),
        'Peak Date': pk['date'].strftime('%Y-%m-%d'),
        'Peak Vol (kAF)': round(pk['swe_volume_kaf'], 1),
        'Peak Melt (mm/d)': round(mlt_pk['mean_melt_mm_day'], 1),
        'Peak Melt Date': mlt_pk['date'].strftime('%Y-%m-%d'),
        'Cum Melt (mm)': round(grp['mean_melt_mm_day'].sum(), 1),
    })

sn_sub_peak_df = pd.DataFrame(sub_rows).sort_values('Peak SWE (mm)', ascending=False).reset_index(drop=True)
print("\nSNODAS 12 HUC-8 Subbasin Summary (WY2023):")
print(sn_sub_peak_df.to_string(index=False))

In [ ]:
# Figure 2: Dual-panel SNODAS Peak SWE and Cumulative WY2023 Snowmelt Runoff (Arial)
# Extract the Peak SWE slice and sum all 196 daily melt bands per pixel
sn_pk_idx = dates_str.index(sn_pk_row['date'].strftime('%Y-%m-%d'))
sn_pk_grid = swe_5070_stack[sn_pk_idx]
sn_cum_melt_grid = np.nansum(melt_5070_stack, axis=0)
sn_cum_melt_grid[~h4_mask] = np.nan  # nansum returns 0.0 on all-NaN pixels outside the basin, so re-mask

x_min, y_min, x_max, y_max = huc4_proj.total_bounds
# Use 99.5th percentile rounded up to keep map contrast clear across the basin
swe_vmax = float(np.ceil(np.nanpercentile(sn_pk_grid, 99.5) / 25.0) * 25.0)
mlt_vmax = float(np.ceil(np.nanpercentile(sn_cum_melt_grid, 99.5) / 25.0) * 25.0)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 6.0), sharey=True)
panels = [
    (axes[0], sn_pk_grid, SNOW_CMAP, swe_vmax, 'Snow Water Equivalent (mm)',
     f"Peak SWE ({sn_pk_row['date'].strftime('%Y-%m-%d')})\nBasin Mean: {sn_pk_row['mean_swe_mm']:.1f} mm ({sn_pk_row['mean_swe_in']:.2f} in) | {sn_pk_row['swe_volume_kaf']:,.0f} kAF"),
    (axes[1], sn_cum_melt_grid, MELT_CMAP, mlt_vmax, 'Cumulative Snow Melt Runoff (mm)',
     f"Cumulative WY2023 Snowmelt Runoff (1044)\nBasin Total: {sn_basin_df['cum_melt_mm'].iloc[-1]:.1f} mm"),
]

# Draw both maps with HUC-8 boundaries, subbasin ID badges, scale bar, and north arrow
for idx, (ax, arr, cmap, vmax, clabel, title) in enumerate(panels):
    im = ax.imshow(arr, cmap=cmap, vmin=0, vmax=vmax, extent=ext_5070)
    huc8_proj.boundary.plot(ax=ax, color='#566573', lw=0.75, alpha=0.85)
    huc4_proj.boundary.plot(ax=ax, color='#0B2545', lw=2.0)
    for _, r in huc8_proj.iterrows():
        pt = r.geometry.representative_point()
        lx, ly = pt.x, pt.y + (18_000 if r['short_id'] == '02' else 0)
        ax.text(lx, ly, r['short_id'], ha='center', va='center', fontsize=8,
                fontweight='bold', color='#1C2833',
                bbox=dict(boxstyle='round,pad=0.14', facecolor='white', edgecolor='none', alpha=0.78))
    ax.set_xlim(x_min - 18_000, x_max + 18_000)
    ax.set_ylim(y_min - 22_000, y_max + 18_000)
    ax.set_aspect('equal')
    add_scalebar(ax, length_km=50, location=(0.05, 0.05))
    add_north_arrow(ax, x_frac=0.93, y_frac=0.87)
    format_km_axes(ax)
    if idx == 1:
        ax.set_ylabel('')  # Hide repeated y-axis label on the right subplot
    ax.set_title(title, fontsize=10.5, pad=8)
    fig.colorbar(im, ax=ax, shrink=0.84, pad=0.02, label=clabel)

fig.suptitle('Minnesota River Basin (HUC 0702): WY2023 SNODAS Peak SWE and Seasonal Melt Runoff',
             fontsize=13, fontweight='bold', x=0.5, y=0.98)
fig.subplots_adjust(left=0.06, right=0.97, top=0.86, bottom=0.12, wspace=0.10)
fig.text(0.01, 0.01,
         'Data: NOAA/NOHRSC SNODAS G02158 (1034 & 1044) | Reprojected: bilinear to EPSG:5070 (1,000 m)',
         fontsize=8.5, color='#566573')
fig2_path = OUTPUT_DIR / 'snodas_fig2_peak_swe_cum_melt_map.png'
fig.savefig(fig2_path, dpi=300)
plt.show()
print(f"Saved: {fig2_path}")

In [ ]:
# Figure 3: SNODAS SWE vs. 24-hr Snowmelt Runoff + Subbasin Melt Summary (Arial)
fig, (ax_top, ax_bot) = plt.subplots(2, 1, figsize=(12.5, 7.8), gridspec_kw={'height_ratios': [1.15, 1.0], 'hspace': 0.34})

# Top subplot: daily SWE curve on left y-axis
l1, = ax_top.plot(sn_basin_df['date'], sn_basin_df['mean_swe_mm'], color='#1B4F72', lw=2.3, label='Basin-Mean SWE (mm, 06:00 UTC)')
ax_top.fill_between(sn_basin_df['date'], 0, sn_basin_df['mean_swe_mm'], color='#5DADE2', alpha=0.22)
ax_top.set_ylabel('Snow Water Equivalent (mm)', color='#1B4F72', fontweight='bold')
ax_top.tick_params(axis='y', labelcolor='#1B4F72')
ax_top.set_ylim(0, sn_basin_df['mean_swe_mm'].max() * 1.2)

# Right twin y-axis on the same top panel for 24-hour snowmelt runoff bars
ax_m = ax_top.twinx()
ax_m.spines['top'].set_visible(False)
bars_m = ax_m.bar(sn_basin_df['date'], sn_basin_df['mean_melt_mm_day'], width=1.0,
                  color='#CA6F1E', alpha=0.70, label='24-hr Snow Melt Runoff (mm/d)')
ax_m.set_ylabel('Snow Melt Runoff (mm/day)', color='#A04000', fontweight='bold')
ax_m.tick_params(axis='y', labelcolor='#A04000')
ax_m.set_ylim(0, sn_basin_df['mean_melt_mm_day'].max() * 1.3)

# Annotate peak SWE date and peak melt runoff day directly on the curves
ax_top.annotate(
    f"Peak SWE: {sn_pk_row['mean_swe_mm']:.1f} mm\n({sn_pk_row['date'].strftime('%Y-%m-%d')})",
    xy=(sn_pk_row['date'], sn_pk_row['mean_swe_mm']),
    xytext=(-110, 12), textcoords='offset points', fontsize=9, fontweight='bold', color='#1B4F72',
    arrowprops=dict(arrowstyle='->', color='#1B4F72', lw=1.1),
    bbox=dict(boxstyle='round,pad=0.28', facecolor='white', edgecolor='#AED6F1')
)
ax_m.annotate(
    f"Peak Melt: {sn_melt_pk_row['mean_melt_mm_day']:.1f} mm/d\n({sn_melt_pk_row['date'].strftime('%Y-%m-%d')})",
    xy=(sn_melt_pk_row['date'], sn_melt_pk_row['mean_melt_mm_day']),
    xytext=(16, 10), textcoords='offset points', fontsize=9, fontweight='bold', color='#A04000',
    arrowprops=dict(arrowstyle='->', color='#A04000', lw=1.1),
    bbox=dict(boxstyle='round,pad=0.28', facecolor='white', edgecolor='#F5CBA7')
)
ax_top.xaxis.set_major_locator(mdates.MonthLocator())
ax_top.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_top.grid(alpha=0.25)
ax_top.legend([l1, bars_m], [l1.get_label(), bars_m.get_label()], loc='upper left', frameon=True)
ax_top.set_title('Minnesota River HUC 4 (0702): Daily SNODAS SWE and 24-Hour Snowmelt Runoff (WY2023)')

# Bottom subplot: grouped horizontal bars comparing each subbasin's Peak SWE vs. Cumulative Melt
sub_plot = sn_sub_peak_df.sort_values('Peak SWE (mm)', ascending=True)
y_pos = np.arange(len(sub_plot))
h = 0.38
ax_bot.barh(y_pos + h / 2, sub_plot['Peak SWE (mm)'], height=h, color='#2E86C1', edgecolor='#1B4F72', label='Peak SWE (mm)')
ax_bot.barh(y_pos - h / 2, sub_plot['Cum Melt (mm)'], height=h, color='#EB984E', edgecolor='#A04000', label='Cumulative WY2023 Melt Runoff (mm)')
sub_max_x = float(max(sub_plot['Peak SWE (mm)'].max(), sub_plot['Cum Melt (mm)'].max()))
ax_bot.set_yticks(y_pos)
ax_bot.set_yticklabels(sub_plot['Subbasin'])
ax_bot.set_xlim(0, sub_max_x * 1.08)
ax_bot.set_xlabel('Water Depth Equivalent (mm)')
ax_bot.set_title('HUC-8 Subbasin Peak SWE vs. Cumulative Melt', loc='left', pad=10)
ax_bot.grid(axis='x', alpha=0.25)
ax_bot.legend(loc='lower right', bbox_to_anchor=(1.0, 1.01), ncol=2, frameon=True)

fig.subplots_adjust(left=0.18, right=0.92, top=0.93, bottom=0.09)
fig.text(0.01, 0.01,
         'Note: Cumulative snowmelt runoff can exceed single-day Peak SWE because mid-winter thaw and late-spring snowfalls also melt.',
         fontsize=8.5, color='#566573')
fig3_path = OUTPUT_DIR / 'snodas_fig3_swe_melt_hydrograph.png'
fig.savefig(fig3_path, dpi=300)
plt.show()
print(f"Saved: {fig3_path}")

## Part 6: Comparing SNODAS (1 km) and UA 800 m Across WY2023

This cell merges Lesson 3's SNODAS time series (`06:00 UTC` snapshot) with Lesson 2's UA 800 m exports (`12:00 UTC` PRISM day), projects UA Peak SWE onto the `EPSG:5070` `1,000 m` grid to compute spatial `SNODAS - UA` differences, and generates Figure 4 (daily curve overlay, difference map, and 12-subbasin scatter). If you opened Lesson 3 without running Lesson 2 in the same session, it reads the reference UA summary CSVs in `data/`.

In [ ]:
ua_basin_file = _locate_local_file('ua_wy2023_huc04_daily_timeseries.csv') or find_course_file('ua_800m_mn_river_wy2023_basin_summary.csv')
ua_sub_file = _locate_local_file('ua_wy2023_huc08_daily_timeseries.csv') or find_course_file('ua_800m_mn_river_wy2023_subbasin_summary.csv')

ua_basin_df = pd.read_csv(ua_basin_file, parse_dates=['date'])
# Read HUC-8 codes as strings (dtype={'huc8': str}) so leading zeros like '07020012' are kept intact
ua_sub_df = pd.read_csv(ua_sub_file, dtype={'huc8': str}, parse_dates=['date'])

# Merge SNODAS and UA daily time series on date and compute daily SNODAS - UA spread
cmp_df = pd.merge(
    sn_basin_df[['date', 'mean_swe_mm', 'mean_swe_in', 'swe_volume_kaf']],
    ua_basin_df[['date', 'mean_swe_mm', 'mean_swe_in']],
    on='date', suffixes=('_snodas', '_ua')
)
cmp_df['diff_snodas_minus_ua_mm'] = cmp_df['mean_swe_mm_snodas'] - cmp_df['mean_swe_mm_ua']

# Calculate bias and RMSE across days where snow cover exists (> 1 mm)
snow_days = cmp_df[(cmp_df['mean_swe_mm_snodas'] > 1.0) | (cmp_df['mean_swe_mm_ua'] > 1.0)]
bias_mm = float(snow_days['diff_snodas_minus_ua_mm'].mean())
rmse_mm = float(np.sqrt(np.mean(snow_days['diff_snodas_minus_ua_mm'] ** 2)))
corr_r = float(cmp_df['mean_swe_mm_snodas'].corr(cmp_df['mean_swe_mm_ua']))
ua_pk_row = cmp_df.loc[cmp_df['mean_swe_mm_ua'].idxmax()]

print(f"HUC-4 Cross-Comparison (196 days, Pearson r = {corr_r:.4f}):")
print(f"  SNODAS Peak SWE : {sn_pk_row['mean_swe_mm']:.1f} mm ({sn_pk_row['mean_swe_in']:.2f} in) on {sn_pk_row['date'].strftime('%Y-%m-%d')}")
print(f"  UA 800m Peak SWE: {ua_pk_row['mean_swe_mm_ua']:.1f} mm ({ua_pk_row['mean_swe_in_ua']:.2f} in) on {ua_pk_row['date'].strftime('%Y-%m-%d')}")
print(f"  Snow-season mean bias (SNODAS - UA): {bias_mm:+.2f} mm | RMSE: {rmse_mm:.2f} mm")

# Put UA Peak SWE onto the exact SNODAS 1,000 m EPSG:5070 grid for pixel-level difference mapping
ua_swe_tif = find_course_file('ua_800m_mn_river_wy2023_swe.tif')
with rasterio.open(ua_swe_tif) as u_src:
    u_dates = list(u_src.descriptions)
    ua_pk_raw = u_src.read(u_dates.index(ua_pk_row['date'].strftime('%Y-%m-%d')) + 1)
    u_crs, u_tr = u_src.crs, u_src.transform

# Reproject UA Peak SWE (-999=NoData) to match SNODAS dst_tr, dst_h, dst_w before subtracting
ua_pk_on_1k = np.full((dst_h, dst_w), np.nan, dtype=np.float32)
reproject(
    np.where(ua_pk_raw == -999, np.nan, ua_pk_raw.astype(np.float32)),
    ua_pk_on_1k,
    src_transform=u_tr, src_crs=u_crs,
    dst_transform=dst_tr, dst_crs=TARGET_CRS,
    src_nodata=np.nan, dst_nodata=np.nan,
    resampling=Resampling.bilinear,
)
ua_pk_on_1k[~h4_mask] = np.nan
peak_spatial_diff = sn_pk_grid - ua_pk_on_1k

# Aggregate Peak SWE per HUC-8 subbasin for both datasets and join on huc8 ID
sn_h8_pk = sn_sub_df.groupby('huc8', as_index=False)['mean_swe_mm'].max().rename(columns={'mean_swe_mm': 'snodas_peak_mm'})
ua_h8_pk = ua_sub_df.groupby('huc8', as_index=False)['mean_swe_mm'].max().rename(columns={'mean_swe_mm': 'ua_peak_mm'})
h8_meta = huc8_proj[['huc8', 'short_id', 'name']].copy()
h8_cmp = h8_meta.merge(sn_h8_pk, on='huc8').merge(ua_h8_pk, on='huc8')
h8_cmp['diff_mm'] = (h8_cmp['snodas_peak_mm'] - h8_cmp['ua_peak_mm']).round(1)
print("\nHUC-8 Peak SWE Comparison (SNODAS vs. UA 800 m):")
print(h8_cmp[['short_id', 'name', 'snodas_peak_mm', 'ua_peak_mm', 'diff_mm']].round(1).to_string(index=False))

# Figure 4: 3-Panel Cross-Dataset Synthesis (Arial)
fig = plt.figure(figsize=(14.5, 8.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.08], hspace=0.35, wspace=0.25)
ax_ov = fig.add_subplot(gs[0, :])   # Wide top panel: daily basin-mean SWE overlay
ax_mp = fig.add_subplot(gs[1, 0])   # Bottom-left: pixel difference map
ax_sc = fig.add_subplot(gs[1, 1])   # Bottom-right: 12-subbasin scatter

# Top panel: plot both curves and shade positive/negative spread
ax_ov.plot(cmp_df['date'], cmp_df['mean_swe_mm_snodas'], color='#1B4F72', lw=2.2, label='SNODAS 1 km (06:00 UTC)')
ax_ov.plot(cmp_df['date'], cmp_df['mean_swe_mm_ua'], color='#117864', lw=2.2, ls='--', label='UA 800 m (12:00 UTC PRISM day)')
ax_ov.fill_between(cmp_df['date'], cmp_df['mean_swe_mm_ua'], cmp_df['mean_swe_mm_snodas'],
                   where=(cmp_df['mean_swe_mm_snodas'] >= cmp_df['mean_swe_mm_ua']),
                   color='#5DADE2', alpha=0.22, label='SNODAS > UA')
ax_ov.fill_between(cmp_df['date'], cmp_df['mean_swe_mm_ua'], cmp_df['mean_swe_mm_snodas'],
                   where=(cmp_df['mean_swe_mm_snodas'] < cmp_df['mean_swe_mm_ua']),
                   color='#EC7063', alpha=0.22, label='UA > SNODAS')
ax_ov.xaxis.set_major_locator(mdates.MonthLocator())
ax_ov.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax_ov.set_ylabel('Basin-Mean SWE (mm)')
ax_ov.set_title(f"Daily Basin-Mean SWE Overlay (r = {corr_r:.3f}, Snow-Season RMSE = {rmse_mm:.1f} mm)")
ax_ov.grid(alpha=0.25)
ax_ov.legend(loc='upper left', ncol=2, frameon=True)

# Bottom-left panel: symmetric diverging palette centered at 0 mm difference
d_lim = float(np.ceil(np.nanpercentile(np.abs(peak_spatial_diff), 98) / 10.0) * 10.0)
im_sp = ax_mp.imshow(peak_spatial_diff, cmap=DIFF_CMAP, vmin=-d_lim, vmax=d_lim, extent=ext_5070)
huc8_proj.boundary.plot(ax=ax_mp, color='#566573', lw=0.75, alpha=0.85)
huc4_proj.boundary.plot(ax=ax_mp, color='#0B2545', lw=1.8)
ax_mp.set_xlim(x_min - 18_000, x_max + 18_000)
ax_mp.set_ylim(y_min - 22_000, y_max + 18_000)
ax_mp.set_aspect('equal')
add_scalebar(ax_mp, length_km=50, location=(0.05, 0.05), show_mid=False)
add_north_arrow(ax_mp, x_frac=0.93, y_frac=0.87)
format_km_axes(ax_mp)
ax_mp.set_title('Spatial Peak SWE Difference (SNODAS - UA, mm)')
fig.colorbar(im_sp, ax=ax_mp, shrink=0.85, pad=0.02, label='SNODAS - UA (mm)')

# Bottom-right panel: scatter plot comparing 12 HUC-8 subbasins against 1:1 agreement
lim_max = max(h8_cmp['snodas_peak_mm'].max(), h8_cmp['ua_peak_mm'].max()) * 1.1
ax_sc.plot([0, lim_max], [0, lim_max], 'k--', lw=1.1, alpha=0.6, label='1:1 Agreement')
ax_sc.scatter(h8_cmp['ua_peak_mm'], h8_cmp['snodas_peak_mm'], s=75, color='#1B4F72', edgecolor='white', lw=0.8, zorder=5)
# Custom point-label offsets keep subbasin ID badges clear of nearby dots
label_offsets = {'01': (-16, 5), '02': (-16, -6), '03': (6, -6), '05': (6, 4), '06': (6, 4), '08': (-16, -9)}
for _, r in h8_cmp.iterrows():
    dx, dy = label_offsets.get(r['short_id'], (6, 4))
    ax_sc.annotate(r['short_id'], (r['ua_peak_mm'], r['snodas_peak_mm']),
                   xytext=(dx, dy), textcoords='offset points', fontsize=8.5, fontweight='bold')
ax_sc.set_xlim(60, lim_max)
ax_sc.set_ylim(60, lim_max)
ax_sc.set_xlabel('UA 800 m Subbasin Peak SWE (mm)')
ax_sc.set_ylabel('SNODAS 1 km Subbasin Peak SWE (mm)')
ax_sc.set_title('HUC-8 Peak SWE: SNODAS vs. UA 800 m (ID Labels 01 to 12)')
ax_sc.grid(alpha=0.25)
ax_sc.legend(loc='upper left', frameon=True)

fig.text(0.01, 0.01,
         'Note: SNODAS snapshots are at 06:00 UTC vs. UA 12:00 UTC PRISM day. Common grid: EPSG:5070 (1,000 m, bilinear).',
         fontsize=8.5, color='#566573')
fig.subplots_adjust(bottom=0.08)
fig4_path = OUTPUT_DIR / 'snodas_fig4_vs_ua_synthesis.png'
fig.savefig(fig4_path, dpi=300)
plt.show()
print(f"Saved: {fig4_path}")

Ask your AI assistant: *"Look at Figure 4's spatial difference map (`SNODAS - UA`) and the 12-subbasin scatter plot. Even though both datasets assimilate ground snow observations, why can SNODAS and UA diverge by 20 to 40 mm in individual Minnesota subbasins? Mention airborne gamma flights, COOP snow density assumptions, and NWP vs. PRISM precipitation."*

## Part 7: Exporting Traceable SNODAS Tables and Peak GeoTIFF

This cell exports daily HUC-4 and HUC-8 SNODAS CSV tables, the subbasin `SNODAS - UA` summary table, and the clipped Peak SWE GeoTIFF (`EPSG:5070`, `1,000 m`) into `OUTPUT_DIR`.

In [ ]:
# Build a provenance string recording dataset, CRS, resampling, and run date
sn_prov = (
    f"NOAA/NOHRSC SNODAS G02158 (1034, 1044); DOI:10.7265/N5TB14TC; "
    f"Target:{TARGET_CRS} {TARGET_RES_M:.0f}m bilinear; Accessed:{datetime.date.today().isoformat()}"
)

sn_basin_out = OUTPUT_DIR / 'snodas_wy2023_huc04_daily_timeseries.csv'
sn_sub_out = OUTPUT_DIR / 'snodas_wy2023_huc08_daily_timeseries.csv'
cmp_out = OUTPUT_DIR / 'snodas_wy2023_vs_ua_comparison.csv'
sn_peak_tif_out = OUTPUT_DIR / 'snodas_wy2023_peak_swe_5070_1000m.tif'

# Copy DataFrames before adding provenance text so analysis variables remain unchanged
b_exp = sn_basin_df.copy()
b_exp['source_note'] = sn_prov
b_exp.to_csv(sn_basin_out, index=False)

s_exp = sn_sub_df.copy()
s_exp['source_note'] = sn_prov
s_exp.to_csv(sn_sub_out, index=False)

h8_cmp.to_csv(cmp_out, index=False)

# Replace NaN with -9999.0 before writing the clipped Peak SWE float32 GeoTIFF
tif_arr = np.where(np.isnan(sn_pk_grid), -9999.0, sn_pk_grid).astype(np.float32)
with rasterio.open(
    sn_peak_tif_out, 'w',
    driver='GTiff', height=dst_h, width=dst_w, count=1, dtype='float32',
    crs=TARGET_CRS, transform=dst_tr, nodata=-9999.0,
    compress='deflate', predictor=2, tiled=True,
) as dst:
    dst.write(tif_arr, 1)
    dst.set_band_description(1, f"SNODAS_SWE_{sn_pk_row['date'].strftime('%Y-%m-%d')}")
    dst.update_tags(
        source='NOAA NOHRSC SNODAS via NSIDC G02158',
        date=sn_pk_row['date'].strftime('%Y-%m-%d'),
        units='millimeters',
        resampling='bilinear',
        provenance=sn_prov,
    )

for p in (sn_basin_out, sn_sub_out, cmp_out, sn_peak_tif_out):
    print(f"Exported: {p} ({p.stat().st_size / 1024:.1f} KB)")

### Optional: Inspect Saved File Locations and Download Lesson 3 Outputs

All Lesson 3 tables, the Peak SWE GeoTIFF, and figures are saved inside `OUTPUT_DIR`:

- In Google Colab, click the **Files** folder icon on the left sidebar and open `/content/module10_outputs/` (or `/content/drive/MyDrive/module10_snow_data/outputs/` when `USE_GOOGLE_DRIVE = True`) to browse and right-click individual files.
- Run the cell below to print every saved file path and size, and package all Lesson 3 outputs into `lesson3_snodas_outputs.zip`.
- Set `DOWNLOAD_OUTPUTS_ZIP = True` before running if you want Colab to trigger a browser download of `lesson3_snodas_outputs.zip`.

In [ ]:
DOWNLOAD_OUTPUTS_ZIP = False  # Set True in Google Colab to download lesson3_snodas_outputs.zip

# Collect all Lesson 3 CSV, GeoTIFF, and figure outputs
output_files = [
    sn_basin_out, sn_sub_out, cmp_out, sn_peak_tif_out,
    fig1_path, fig2_path, fig3_path, fig4_path,
]

print("Lesson 3 saved files:")
for out_f in output_files:
    assert out_f.exists(), f"Missing output: {out_f}"
    print(f"  {out_f.resolve()} ({out_f.stat().st_size / 1024:,.1f} KB)")

# Compress all outputs into a single zip archive inside OUTPUT_DIR
zip_path = OUTPUT_DIR / 'lesson3_snodas_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for out_f in output_files:
        zf.write(out_f, arcname=out_f.name)
print(f"\nPackaged zip archive: {zip_path.resolve()} ({zip_path.stat().st_size / 1024:,.1f} KB)")

# Trigger a browser download when running inside Google Colab
if DOWNLOAD_OUTPUTS_ZIP:
    if IN_COLAB:
        from google.colab import files as colab_files
        colab_files.download(str(zip_path))
    else:
        print("Running locally: copy or open the .zip path printed above.")

## Part 8: Practice Debugging 1: Filling Outage Dates With Zero

NSIDC's `G02158_missing_files.txt` lists 18 masked outage dates between 2004 and 2012 (including winter dates `2007-03-26` and `2008-03-13`). A common bug when analyzing multi-year SNODAS series is filling missing dates with `0.0` before plotting or computing daily `diff()`.

The cell below drops `2023-03-26` and `2023-03-27` from the WY2023 series and applies `.fillna(0.0)`, creating a false `105 mm/day` melt spike and `104 mm/day` accumulation spike without raising an error.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING 1: THIS CELL CONTAINS AN INTENTIONAL BUG.
# It simulates a 2-day NSIDC archive outage and replaces missing
# days with 0.0 before computing daily SWE change and monthly mean.
# ---------------------------------------------------------------
mar_apr = sn_basin_df[(sn_basin_df['date'] >= '2023-03-15') & (sn_basin_df['date'] <= '2023-04-15')].copy()
outage_dates = [pd.Timestamp('2023-03-26'), pd.Timestamp('2023-03-27')]

observed = mar_apr[~mar_apr['date'].isin(outage_dates)].set_index('date')['mean_swe_mm']
full_calendar = pd.date_range('2023-03-15', '2023-04-15', freq='D')

# Bug: filling missing archive dates with 0.0
buggy_series = observed.reindex(full_calendar).fillna(0.0)
buggy_dswe = buggy_series.diff()

true_series = mar_apr.set_index('date')['mean_swe_mm']
true_dswe = true_series.diff()

print(f"Buggy March-April mean SWE : {buggy_series.mean():.2f} mm  (true: {true_series.mean():.2f} mm)")
print(f"Buggy max 1-day 'melt'     : {buggy_dswe.min():.1f} mm/d  (true max 1-day melt: {true_dswe.min():.1f} mm/d)")
print(f"Buggy max 1-day 'blizzard' : {buggy_dswe.max():+.1f} mm/d  (true max 1-day gain: {true_dswe.max():+.1f} mm/d)")

fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.plot(true_series.index, true_series.values, color='#117864', lw=2.0, ls='--', label='True Continuous SWE')
ax.plot(buggy_series.index, buggy_series.values, color='#B03A2E', lw=2.0, label='Buggy .fillna(0.0) Series')
ax.set_title('Silent Missing-Day Bug: Fake Melt and Fake Blizzard from .fillna(0.0)')
ax.set_ylabel('Basin-Mean SWE (mm)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.grid(alpha=0.25)
ax.legend(frameon=True)
plt.show()

### Your Task (1)

1. Explain why setting two missing March dates to `0.0` produced a `-105.5 mm/day` drop followed by a `+103.8 mm/day` jump in `diff()`.
2. Record missing dates using a boolean `is_interpolated` column and linearly interpolate gaps up to 5 consecutive days without `.fillna(0.0)`.

Suggested AI prompt:

*"Here is a daily SNODAS SWE series missing `2023-03-26` and `2023-03-27`. Replacing missing dates with `0.0` created fake melt and accumulation spikes in `diff()`. Write a pandas repair function that keeps an `is_interpolated` flag, linearly interpolates ONLY gaps `<= 5` consecutive days, and recalculates `mean` and `diff()`."*

In [ ]:
# YOUR FIX: reindex to full_calendar, track is_interpolated, and interpolate short gaps (never fillna(0)).
#
# Hints:
# 1. Start from observed.reindex(full_calendar) WITHOUT .fillna(0.0).
# 2. Record is_interpolated = observed.reindex(full_calendar).isna(), then call .interpolate(method='linear', limit=5, limit_area='inside').
# 3. Verify that max 1-day melt and gain match physical snowpack behavior.

## Part 9: Practice Debugging 2: `100x` Melt Scale Error and Unaligned 2013 Grids

This cell demonstrates two bugs that execute without raising warnings:

- Multiplying 24-hour Snow Melt Runoff (`1044`) by `1.0` (the SWE factor `/ 1,000`) in place of `0.01` (`/ 100,000`), inflating daily melt by `100x`.
- Calling `calculate_default_transform()` separately on `2013-09-30` and `2013-10-01`, producing shifted output grid origins (`ΔX = 34.49 m, ΔY = 47.19 m`) that prevent pixel-wise subtraction.

In [ ]:
# ---------------------------------------------------------------
# PRACTICE DEBUGGING 2: THIS CELL CONTAINS TWO INTENTIONAL BUGS.
# ---------------------------------------------------------------
# Bug A: Using SWE's scale factor (raw * 1.0 = mm) on Snow Melt Runoff (1044)
wrong_melt_mm = reproject_snodas_slice(
    melt_demo_raw, sn_crs, sn_tr, scale_to_mm=1.0, resampling=Resampling.bilinear, land_valid_mask=swe_land_demo
)
buggy_melt_mean = float(np.nanmean(wrong_melt_mm))
true_melt_mean = float(np.nanmean(demo_melt_grids['bilinear']))
basin_peak_swe = float(sn_pk_row['mean_swe_mm'])

print(f"Melt date {melt_demo_date}:")
print(f"  Buggy 1-day melt runoff : {buggy_melt_mean:,.1f} mm/day  (vs Peak SWE of {basin_peak_swe:.1f} mm)")
print(f"  True 1-day melt runoff  : {true_melt_mean:.2f} mm/day  (factor error = {buggy_melt_mean / max(true_melt_mean, 1e-6):.0f}x)")

# Bug B: Calling calculate_default_transform() separately on pre- and post-2013 subsets
with (
    rasterio.open(find_course_file('snodas_20130930_swe_subset.tif')) as pre_s,
    rasterio.open(find_course_file('snodas_20131001_swe_subset.tif')) as post_s,
):
    tr_pre_auto, w_pre, h_pre = calculate_default_transform(pre_s.crs, TARGET_CRS, pre_s.width, pre_s.height, *pre_s.bounds)
    tr_post_auto, w_post, h_post = calculate_default_transform(post_s.crs, TARGET_CRS, post_s.width, post_s.height, *post_s.bounds)
    print("\nUnaligned default transforms in EPSG:5070:")
    print(f"  2013-09-30 UL: ({tr_pre_auto.c:.2f}, {tr_pre_auto.f:.2f}) | dx={tr_pre_auto.a:.2f} m | shape=({h_pre}, {w_pre})")
    print(f"  2013-10-01 UL: ({tr_post_auto.c:.2f}, {tr_post_auto.f:.2f}) | dx={tr_post_auto.a:.2f} m | shape=({h_post}, {w_post})")
    print(f"  Origin mismatch: ΔX = {tr_post_auto.c - tr_pre_auto.c:.2f} m, ΔY = {tr_post_auto.f - tr_pre_auto.f:.2f} m")

### Your Task (2)

1. Check why a single day of melt runoff printed `1,745.4 mm/day` against a `117.4 mm` snowpack using `.txt` `Data units` (`Meters / 100000.0`).
2. Reproject both `2013-09-30` and `2013-10-01` onto the shared snapped grid `(dst_tr, dst_h, dst_w)` and verify exact transform and shape alignment.

Suggested AI prompt:

*"My SNODAS 24-hour Snow Melt (`1044`) grid reports hundreds of mm/day of runoff from a ~117 mm snowpack, and my `2013-09-30` and `2013-10-01` reprojected rasters have shifted origins. Fix both using `scale_to_mm=0.01` and a shared snapped `EPSG:5070` grid."*

In [ ]:
# YOUR FIX: apply scale_to_mm=0.01 for product 1044 and reproject both 2013 dates to snapped (dst_tr, dst_h, dst_w).
#
# Hints:
# 1. Pass scale_to_mm=0.01 in reproject_snodas_slice(...) and verify 1-day melt < Peak SWE.
# 2. Reproject both 2013-09-30 and 2013-10-01 onto dst_tr and assert identical shapes and transforms.

## Engineering Interpretation and Choosing Between UA 800 m and SNODAS 1 km

| Decision Factor | University of Arizona (`DailyData_800m`) | NOAA/NOHRSC SNODAS (`G02158`) |
|---|---|---|
| Record length | WY1982 (Oct 1981) to present (~44+ years for long drought/flood frequency studies) | Sep 30, 2003 to present (routine assimilation from Oct 2004, ~21+ years) |
| Spatial grid and format | ~800 m (30 arc-sec), `EPSG:4269` (NAD83), NetCDF4 | ~1 km (30 arc-sec), `EPSG:4326` (WGS84), raw binary `.dat` and `.txt` |
| Available variables | `SWE`, `Snow Depth` | 8 products: `SWE`, `Snow Depth`, `Snow Melt Runoff`, `Sublimation`, `Blowing Snow`, `Solid/Liquid Precip`, `Snowpack Temp` |
| Observation timing | PRISM day ending `12:00 UTC` (`06:00 CST` / `07:00 CDT`) | State snapshot at `06:00 UTC` (`00:00 CST` / `01:00 CDT`), fluxes `06:00-06:00 UTC` |
| Assimilation inputs | PRISM + COOP + SNOTEL methodology across 40+ years | Assimilates NOHRSC airborne gamma surveys, satellite SCA, and ground data with hourly physics |
| Known caveats | `_FillValue = -999`, `early`/`provisional` updates, `float32` `lon`/`lat`, SW-US lidar ANN ratio | Missing days (`G02158_missing_files.txt`), `2013-10-01` 5% shift, `2014 to 2019` 126,950 zero cells, `32,767` cap, RUC2 to RAP/HRRR change |

### Selection Guidelines

- Choose UA 800 m when you need a multi-decade baseline (1982 to present) or consistent snow depth and SWE climatology prior to 2005.
- Choose SNODAS when you need operational post-2004 calibration, direct 24-hour snowmelt runoff (`1044`) or rain-on-snow (`1025`) inputs, or Midwest airborne gamma-assimilated SWE during spring flood modeling, and compare basin snow volume against USGS streamgage runoff.

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---|---|---|
| Pre-2013 and post-2013 rasters do not overlap identically | `2013-10-01` 5% cell shift or separate `calculate_default_transform()` calls | Assign rational pre/post-2013 bounds and reproject onto one snapped `EPSG:5070` grid |
| Melt runoff (`1044`) is 100x too big or 100x too small | Divided by `1,000` (SWE scale) instead of `100,000` (`* 0.01` to mm) | Verify `.txt` `Data units` (`Meters / 100000`) |
| Snow-free days turn basin-mean Melt (`1044`) into `nan` or inflated averages | SNODAS `1044` stores snow-free land as `-9999` | Pass `SWE (1034) != -9999` as valid land mask and set land `1044 == -9999` to `0.0 mm/day` |
| Lake or coastal pixels stay `0.0` (`2014-10-09` to `2019-10-10`) | NOHRSC land/water mask initialization bug | Apply NSIDC's `SNODAS_Zero_Repair_Mask.tif` |
| Large `-100+ mm` drops in multi-year time series | Used `.fillna(0.0)` on missing outage dates | Check `G02158_missing_files.txt` and keep missing dates as `NaN` or flagged short interpolation |

## Practice Exercises

### Exercise 1: Mass-Balance Check (`-ΔSWE` vs. Modeled Snowmelt Runoff `1044`)

Across the main spring melt window (`2023-04-01` through `2023-04-20`), compare total SWE loss (`SWE_start - SWE_end`) against cumulative SNODAS modeled Snow Melt Runoff (`1044`) in `sn_basin_df`. Explain why cumulative melt runoff differs slightly from net `ΔSWE` (consider April snowfall, refreezing, sublimation `1050`/`1039`, and assimilation updates).

In [ ]:
# EXERCISE 1: Compare April 1-20 net SWE loss against cumulative SNODAS 1044 melt runoff
# Your code here.

### Exercise 2: Investigate the Largest SNODAS vs. UA Subbasin Discrepancy

From `h8_cmp`, find which of the 12 HUC-8 subbasins has the largest absolute Peak SWE difference (`|SNODAS - UA|`). Plot the full 196-day daily SWE curves of both products for that subbasin in Arial font and inspect whether the difference starts during winter accumulation, peak timing, or spring melt.

In [ ]:
# EXERCISE 2: Plot full WY2023 SNODAS vs. UA SWE curves for the HUC-8 with the largest peak difference
# Your code here.

### Exercise 3: Map Peak Daily Snowmelt Intensity (`2023-04-12`) Across HUC-8 Subbasins

Extract the `2023-04-12` slice from `melt_5070_stack` (the peak basin-mean snowmelt day of WY2023 at `30.3 mm/day`), compute the subbasin-mean 24-hour melt rate for each of the 12 HUC-8s, and map the spatial melt grid in `EPSG:5070` with HUC-8 boundaries, scale bar, and north arrow.

In [ ]:
# EXERCISE 3: Spatial and subbasin map of peak 24-hour snowmelt runoff on 2023-04-12
# Your code here.

## Module 10 Summary

Across Lessons 1, 2, and 3, you completed an end-to-end snow data workflow for the Minnesota River Basin (`HUC 0702`, `44,050.3 km²`):

1. Lesson 1 downloaded USGS WBD boundaries, UA `DailyData_800m` NetCDFs (with `float64` full-span coordinate math), and raw NSIDC SNODAS `.tar` archives (`>i2` + `.txt`).
2. Lesson 2 projected UA 800 m grids onto `EPSG:5070` (`800 m`), yielding WY2023 Peak SWE of `123.7 mm` (`4.87 in`, `5.450 km³`, `4,418 kAF`) on `2023-04-01`.
3. Lesson 3 aligned SNODAS 1 km grids onto `EPSG:5070` (`1,000 m`), yielding WY2023 Peak SWE of `117.4 mm` (`4.62 in`, `4,193 kAF`) on `2023-03-12`, Peak Melt of `30.3 mm/day` on `2023-04-12`, `125.7 mm` seasonal melt runoff, and cross-dataset agreement (`r = 0.9725`, bias `+3.15 mm`, `RMSE = 11.00 mm`).

### Documentation and Citations

- National Operational Hydrologic Remote Sensing Center (2004), *Snow Data Assimilation System (SNODAS) Data Products at NSIDC, Version 1*, Boulder, CO USA: NSIDC, [https://doi.org/10.7265/N5TB14TC](https://doi.org/10.7265/N5TB14TC)
- SNODAS Landing Page and User Guide (rev. Apr 2025): [nsidc.org/data/g02158/versions/1](https://nsidc.org/data/g02158/versions/1) and [g02158-v001-userguide_2_1.pdf](https://nsidc.org/sites/default/files/g02158-v001-userguide_2_1.pdf)
- NSIDC Special Report 11: Barrett, A. (2003), [nsidc_special_report_11.pdf](https://nsidc.org/sites/default/files/nsidc_special_report_11.pdf)
- SNODAS Archive, Errata, and Zero Repair Mask: [G02158/masked/](https://noaadata.apps.nsidc.org/NOAA/G02158/masked/), [G02158_missing_files.txt](https://noaadata.apps.nsidc.org/NOAA/G02158/G02158_missing_files.txt), and [SNODAS_Zero_Repair_Mask.tif](https://noaadata.apps.nsidc.org/NOAA/G02158/ancillary/SNODAS_Zero_Repair_Mask.tif)
- NOAA NOHRSC National Snow Analyses: [nohrsc.noaa.gov/nsa/](https://www.nohrsc.noaa.gov/nsa/)
- UA SWE Server and Citations: [climate.arizona.edu/data/UA_SWE/DailyData_800m/](https://climate.arizona.edu/data/UA_SWE/DailyData_800m/), [NSIDC-0719 (DOI: 10.5067/0GGPB220EX6A)](https://doi.org/10.5067/0GGPB220EX6A), Broxton et al. (2019, 2024)
- USGS Watershed Boundary Dataset: [usgs.gov/national-hydrography/watershed-boundary-dataset](https://www.usgs.gov/national-hydrography/watershed-boundary-dataset)